In [1]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Tạo thư mục ở project root thay vì trong notebooks/
os.makedirs('../models', exist_ok=True)
os.makedirs('../results', exist_ok=True)

In [9]:
import sys
import importlib

# Lùi lại 1 thư mục để truy cập vào src/models/
sys.path.append('../src/models')

# 1. ÉP PYTHON NẠP LẠI (RELOAD) FILE CONFIG VÀ ARCHITECTURE
import model_config_hands
import architectures_hands

importlib.reload(model_config_hands)
importlib.reload(architectures_hands)

# 2. Import trực tiếp các biến và hàm sau khi đã reload
from model_config_hands import INPUT_SHAPE, NUM_CLASSES
from architectures_hands import build_lstm_model, build_bilstm_model, build_cnn1d_model

print(f"Cấu hình TỪ HANDS CONFIG: INPUT_SHAPE={INPUT_SHAPE}, NUM_CLASSES={NUM_CLASSES}")

Cấu hình TỪ HANDS CONFIG: INPUT_SHAPE=(30, 126), NUM_CLASSES=30


In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_and_print(model, model_name, X_test, y_test):
    # Lấy dự đoán từ mô hình
    y_pred_probs = model.predict(X_test, verbose=0)
    # Chuyển đổi xác suất thành nhãn (Label Encoding)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Tính toán các chỉ số đánh giá
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    # In kết quả theo format định dạng sẵn
    print("=================================================================")
    print(f"           {model_name.upper()} EVALUATION RESULTS (TEST SET)")
    print("=================================================================")
    print(f"Accuracy:          {acc * 100:.2f}%")
    print(f"Macro Precision:   {precision:.4f}")
    print(f"Macro Recall:      {recall:.4f}")
    print(f"Macro F1-score:    {f1:.4f}")
    print("=================================================================")

In [11]:
# 3. Load Data từ thư mục hands
data_dir = "../data/hands/" 
X_train = np.load(f"{data_dir}X_train_aug.npy")
y_train = np.load(f"{data_dir}y_train_aug.npy")
X_val = np.load(f"{data_dir}X_val.npy")
y_val = np.load(f"{data_dir}y_val.npy")
X_test = np.load(f"{data_dir}X_test.npy")
y_test = np.load(f"{data_dir}y_test.npy")

print("\n--- KIỂM TRA SHAPE DỮ LIỆU ---")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

# Đảm bảo nhãn đang ở dạng mảng 1 chiều (Label Encoding)
assert len(y_train.shape) == 1, "LỖI: Nhãn không phải Label Encoding 1 chiều!"
print("\n-> [OK] Dữ liệu đã sẵn sàng để huấn luyện đa mô hình!")


--- KIỂM TRA SHAPE DỮ LIỆU ---
X_train shape: (3000, 30, 126)
y_train shape: (3000,)

-> [OK] Dữ liệu đã sẵn sàng để huấn luyện đa mô hình!


In [12]:
# CẤU HÌNH HUẤN LUYỆN CHUNG
EPOCHS = 50
BATCH_SIZE = 32

In [14]:
print("\n--- XÂY DỰNG & HUẤN LUYỆN LSTM (HANDS) ---")
lstm_model = build_lstm_model()

lstm_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    # ĐÃ SỬA: Cập nhật tên file thành lstm_model_hand.h5
    ModelCheckpoint('../models/lstm_model_hand.h5', monitor='val_loss', save_best_only=True)
]

lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=lstm_callbacks,
    verbose=1
)

# ĐÃ SỬA: Cập nhật tên file thành lstm_history_hand.json
with open('../results/lstm_history_hand.json', 'w') as f:
    json.dump(lstm_history.history, f)

print(f"\n[*] LSTM training completed.")
print(f"[*] Final validation accuracy: {lstm_history.history['val_accuracy'][-1]:.4f}")
print("[*] Model saved to: ../models/lstm_model_hand.h5")


--- XÂY DỰNG & HUẤN LUYỆN LSTM (HANDS) ---
Epoch 1/50
94/94 [==============================] - 10s 54ms/step - loss: 2.6561 - accuracy: 0.2397 - val_loss: 1.6077 - val_accuracy: 0.5810
Epoch 2/50
 1/94 [..............................] - ETA: 5s - loss: 1.7236 - accuracy: 0.5000

d:\anaconda3\envs\sign_lang_env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


94/94 [==============================] - 4s 44ms/step - loss: 1.4522 - accuracy: 0.5760 - val_loss: 1.1127 - val_accuracy: 0.7286
Epoch 3/50
94/94 [==============================] - 4s 42ms/step - loss: 1.0945 - accuracy: 0.6807 - val_loss: 1.0045 - val_accuracy: 0.7190
Epoch 4/50
94/94 [==============================] - 4s 41ms/step - loss: 0.8692 - accuracy: 0.7467 - val_loss: 1.0107 - val_accuracy: 0.7238
Epoch 5/50
94/94 [==============================] - 4s 43ms/step - loss: 0.6761 - accuracy: 0.8020 - val_loss: 0.8269 - val_accuracy: 0.7857
Epoch 6/50
94/94 [==============================] - 4s 46ms/step - loss: 0.6329 - accuracy: 0.8220 - val_loss: 0.7313 - val_accuracy: 0.8286
Epoch 7/50
94/94 [==============================] - 4s 43ms/step - loss: 0.4738 - accuracy: 0.8733 - val_loss: 0.8311 - val_accuracy: 0.8095
Epoch 8/50
94/94 [==============================] - 4s 44ms/step - loss: 0.4381 - accuracy: 0.8870 - val_loss: 0.7799 - val_accuracy: 0.8190
Epoch 9/50
94/94 [======

In [15]:
# GỌI HÀM ĐÁNH GIÁ (Xuất bảng kết quả Test Set)
evaluate_and_print(lstm_model, "LSTM (HANDS)", X_test, y_test)

           LSTM (HANDS) EVALUATION RESULTS (TEST SET)
Accuracy:          86.73%
Macro Precision:   0.8867
Macro Recall:      0.8637
Macro F1-score:    0.8612


In [16]:
print("\n--- XÂY DỰNG & HUẤN LUYỆN BiLSTM (HANDS) ---")
bilstm_model = build_bilstm_model()

# ĐÃ XÓA: dòng bilstm_model.compile(...) vì đã được gọi bên trong hàm build_bilstm_model()

bilstm_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    # ĐÃ SỬA: Cập nhật tên file thành bilstm_model_hand.h5
    ModelCheckpoint('../models/bilstm_model_hand.h5', monitor='val_loss', save_best_only=True)
]

bilstm_history = bilstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=bilstm_callbacks,
    verbose=1
)

# ĐÃ SỬA: Cập nhật tên file thành bilstm_history_hand.json
with open('../results/bilstm_history_hand.json', 'w') as f:
    json.dump(bilstm_history.history, f)

print(f"\n[*] BiLSTM training completed.")
print(f"[*] Final validation accuracy: {bilstm_history.history['val_accuracy'][-1]:.4f}")
print("[*] Model saved to: ../models/bilstm_model_hand.h5")


--- XÂY DỰNG & HUẤN LUYỆN BiLSTM (HANDS) ---
Epoch 1/50
94/94 [==============================] - 22s 140ms/step - loss: 2.4153 - accuracy: 0.3207 - val_loss: 1.4861 - val_accuracy: 0.5905
Epoch 2/50
 1/94 [..............................] - ETA: 11s - loss: 1.0984 - accuracy: 0.7500

d:\anaconda3\envs\sign_lang_env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


94/94 [==============================] - 7s 76ms/step - loss: 1.2825 - accuracy: 0.6253 - val_loss: 0.9461 - val_accuracy: 0.7286
Epoch 3/50
94/94 [==============================] - 8s 80ms/step - loss: 0.9016 - accuracy: 0.7287 - val_loss: 0.8122 - val_accuracy: 0.7571
Epoch 4/50
94/94 [==============================] - 7s 79ms/step - loss: 0.7385 - accuracy: 0.7873 - val_loss: 0.7114 - val_accuracy: 0.8429
Epoch 5/50
94/94 [==============================] - 9s 97ms/step - loss: 0.5897 - accuracy: 0.8277 - val_loss: 0.7037 - val_accuracy: 0.7857
Epoch 6/50
94/94 [==============================] - 8s 80ms/step - loss: 0.4692 - accuracy: 0.8693 - val_loss: 0.7128 - val_accuracy: 0.8429
Epoch 7/50
94/94 [==============================] - 8s 83ms/step - loss: 0.3653 - accuracy: 0.8933 - val_loss: 0.6073 - val_accuracy: 0.8571
Epoch 8/50
94/94 [==============================] - 8s 81ms/step - loss: 0.3886 - accuracy: 0.8863 - val_loss: 0.6197 - val_accuracy: 0.8571
Epoch 9/50
94/94 [======

In [17]:
# GỌI HÀM ĐÁNH GIÁ (Xuất bảng kết quả Test Set)
evaluate_and_print(bilstm_model, "BiLSTM (HANDS)", X_test, y_test)

           BILSTM (HANDS) EVALUATION RESULTS (TEST SET)
Accuracy:          84.83%
Macro Precision:   0.8592
Macro Recall:      0.8421
Macro F1-score:    0.8400


In [19]:
print("\n--- XÂY DỰNG & HUẤN LUYỆN 1D-CNN (HANDS) ---")
cnn1d_model = build_cnn1d_model()

# ĐÃ XÓA: dòng cnn1d_model.compile(...) vì đã được gọi bên trong hàm build_cnn1d_model()

cnn1d_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    # ĐÃ SỬA: Cập nhật tên file thành cnn1d_model_hand.h5
    ModelCheckpoint('../models/cnn1d_model_hand.h5', monitor='val_loss', save_best_only=True)
]

cnn1d_history = cnn1d_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cnn1d_callbacks,
    verbose=1
)

# ĐÃ SỬA: Cập nhật tên file thành cnn1d_history_hand.json
with open('../results/cnn1d_history_hand.json', 'w') as f:
    json.dump(cnn1d_history.history, f)

print(f"\n[*] 1D-CNN training completed.")
print(f"[*] Final validation accuracy: {cnn1d_history.history['val_accuracy'][-1]:.4f}")
print("[*] Model saved to: ../models/cnn1d_model_hand.h5")


--- XÂY DỰNG & HUẤN LUYỆN 1D-CNN (HANDS) ---
Epoch 1/50
94/94 [==============================] - 3s 13ms/step - loss: 2.3131 - accuracy: 0.3720 - val_loss: 2.0918 - val_accuracy: 0.6381
Epoch 2/50
94/94 [==============================] - 1s 12ms/step - loss: 1.3508 - accuracy: 0.6283 - val_loss: 1.1650 - val_accuracy: 0.8000
Epoch 3/50
94/94 [==============================] - 1s 10ms/step - loss: 1.0221 - accuracy: 0.7157 - val_loss: 0.8030 - val_accuracy: 0.8048
Epoch 4/50
94/94 [==============================] - 1s 11ms/step - loss: 0.8004 - accuracy: 0.7803 - val_loss: 0.7275 - val_accuracy: 0.7762
Epoch 5/50
94/94 [==============================] - 1s 11ms/step - loss: 0.6610 - accuracy: 0.8167 - val_loss: 0.6636 - val_accuracy: 0.8381
Epoch 6/50
94/94 [==============================] - 1s 12ms/step - loss: 0.6030 - accuracy: 0.8367 - val_loss: 0.5185 - val_accuracy: 0.8619
Epoch 7/50
94/94 [==============================] - 1s 14ms/step - loss: 0.5038 - accuracy: 0.8623 - val_los

In [20]:
# GỌI HÀM ĐÁNH GIÁ (Xuất bảng kết quả Test Set)
evaluate_and_print(cnn1d_model, "1D-CNN (HANDS)", X_test, y_test)

           1D-CNN (HANDS) EVALUATION RESULTS (TEST SET)
Accuracy:          89.57%
Macro Precision:   0.9120
Macro Recall:      0.8937
Macro F1-score:    0.8935


In [22]:
from tensorflow.keras.models import load_model
import os # Import os để sử dụng os.path.exists

print("\n--- KIỂM TRA MODEL ĐÃ LƯU ---")
saved_models = {
    # ĐÃ SỬA: Sửa _hands thành _hand để khớp với tên file đã lưu
    "LSTM": "../models/lstm_model_hand.h5",
    "BiLSTM": "../models/bilstm_model_hand.h5",
    "1D-CNN": "../models/cnn1d_model_hand.h5"
}

# Lấy 1 sample từ tập X_test để chạy thử
sample = X_test[:1]

for name, path in saved_models.items():
    if os.path.exists(path):
        # Tải mô hình lên
        test_model = load_model(path)
        
        # Cho mô hình dự đoán thử 1 mẫu
        pred = test_model.predict(sample, verbose=0)
        
        # Kiểm tra xem đầu ra có đúng là 30 class không
        assert pred.shape == (1, NUM_CLASSES), f"Lỗi shape ở {name}"
        
        print(f"-> [OK] {name} tải thành công. Output Shape: {pred.shape}")
    else:
        print(f"-> [LỖI] Không tìm thấy file {path}")


--- KIỂM TRA MODEL ĐÃ LƯU ---
-> [OK] LSTM tải thành công. Output Shape: (1, 30)
-> [OK] BiLSTM tải thành công. Output Shape: (1, 30)
-> [OK] 1D-CNN tải thành công. Output Shape: (1, 30)
